# Sensitivity Analysis - NetLogo BehaviorSpace

Loops through all `*spreadsheet*.csv` files and computes:
- **η²** (eta-squared): effect size
- **OLS slope**: direction and magnitude of change per unit IV increase
- **R²**: goodness of fit of the linear trend

| η² | Interpretation |
|---|---|
| < 0.06 | weak / no effect |
| 0.06 – 0.14 | moderate effect |
| > 0.14 | strong effect |

| Slope sign | Direction |
|---|---|
| `+` | DV increases as IV increases |
| `−` | DV decreases as IV increases |

## 1 · Configuration

In [ ]:
# ── Input / Output ────────────────────────────────────────────────────────────
INPUT_DIR  = "output/sensitivity-analysis"   # folder containing all CSV files
OUTPUT_DIR = "output/output-effects"   # where to save results

# ── CSV structure (same for all BehaviorSpace spreadsheet exports) ────────────
PARAM_ROW      = 8    # row containing parameter name + values per run
DATA_START_ROW = 17   # first data row in [all run data] section

# ── Stabilization period ──────────────────────────────────────────────────────
# Number of initial ticks to exclude before computing averages.
# Set to 0 to include all ticks.
STABILIZATION_TICKS = 500

# ── Map experiment parameter names → readable labels ─────────────────────────
# Key = substring found in the CSV parameter name (case-insensitive)
# Value = human-readable label for plots
PARAM_LABEL_MAP = {
    "headcount-mean":             "Headcount Mean",
    "coaching-turnover-decrease": "Coaching Turnover Decrease",
    "coaching-skill-ceiling":     "Coaching Skill Ceiling",
    "coaching-skill-increase":    "Coaching Skill Increase",
    "working-skill-increase":     "Working Skill Increase",
    "working-turnover-increase":  "Working Turnover Increase",
    "hiring-threshold-mean":      "Hiring Threshold Mean",
    "leaving-threshold":          "Leaving Threshold",
    "skill-decay-rate":           "Skill Decay Rate",
    "max-pxcor":                  "Number of Companies",
}

# ── Dependent variables — from dependent-variables list ───────────────────────
# NetLogo reporter order: [cumulative_revenue, coaching_rate, vacancies, propensity_to_leave]
DEP_VAR_SPECS = [
    ("cumulative_revenue",  0),
    ("coaching_rate",       1),
    ("vacancies",           2),
    ("propensity_to_leave", 3),
]

# ── Dependent variables — from skill-distribution column ──────────────────────
# NetLogo reporter order: [mean, std, median]
SKILL_DIST_SPECS = [
    ("skill_mean", 0),
]

# Set to True to include unemployment_rate as a dependent variable
INCLUDE_UNEMPLOYMENT = True


## 2 · Imports

In [ ]:
import re, ast, csv, io, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
from scipy import stats
matplotlib.use('Agg')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 3 · Helper functions

In [ ]:
def parse_netlogo_list(raw: str) -> list:
    """Convert a NetLogo list string '[1 2 3]' to a Python list."""
    cleaned = re.sub(r"\s+", ", ", raw.strip()).replace("[, ", "[").replace(", ]", "]")
    return ast.literal_eval(cleaned)


def detect_param(csv_path: str) -> tuple[str, str]:
    raw_lines = open(csv_path).readlines()
    param_row = list(csv.reader(io.StringIO(raw_lines[PARAM_ROW - 1])))[0]
    raw_param = param_row[0].strip()
    for key, label in PARAM_LABEL_MAP.items():
        if key.lower() in raw_param.lower():
            return raw_param, label
    return raw_param, raw_param


def load_final_tick_data(csv_path: str) -> pd.DataFrame:
    """Parse BehaviorSpace CSV and return one row per run with averages over all ticks.

    Ticks <= STABILIZATION_TICKS are excluded so the model can reach a stable
    state before averages are computed.
    """
    raw_lines = open(csv_path).readlines()
    param_row = list(csv.reader(io.StringIO(raw_lines[PARAM_ROW - 1])))[0]
    iv_values = {}
    run_number = 1
    for cell in param_row[1:]:
        val = cell.strip()
        if val:
            iv_values[run_number] = float(val)
            run_number += 1
    n_runs = len(iv_values)
    col_names = ["_dummy"]
    for r in range(1, n_runs + 1):
        col_names += [f"step_r{r}", f"dep_vars_r{r}", f"skill_dist_r{r}", f"unemployment_r{r}"]
    df_raw = pd.read_csv(
        csv_path, skiprows=DATA_START_ROW - 1, header=None,
        names=col_names, quotechar='"', low_memory=False,
    )
    records = []
    for r, iv_val in iv_values.items():
        step_col = f"step_r{r}"
        dep_col = f"dep_vars_r{r}"
        skill_col = f"skill_dist_r{r}"
        unemployment_col = f"unemployment_r{r}"
        sub = df_raw[[step_col, dep_col, skill_col, unemployment_col]].dropna(subset=[step_col])
        sub = sub[sub[step_col].apply(lambda x: str(x).strip().lstrip("-").isdigit())].copy()
        sub[step_col] = sub[step_col].astype(int)
        if STABILIZATION_TICKS > 0:
            sub = sub[sub[step_col] > STABILIZATION_TICKS]
        record = {"run": r, "iv_value": iv_val}
        dv_accum = {name: [] for name, _ in DEP_VAR_SPECS}
        skill_accum = {name: [] for name, _ in SKILL_DIST_SPECS}
        unemployment_vals = []
        for _, row in sub.iterrows():
            dep_lists = parse_netlogo_list(str(row[dep_col]))
            if dep_lists is not None:
                for name, idx in DEP_VAR_SPECS:
                    dv_accum[name].append(dep_lists[idx])
            skill_list = parse_netlogo_list(str(row[skill_col]))
            if skill_list is not None:
                for name, idx in SKILL_DIST_SPECS:
                    skill_accum[name].append(float(skill_list[idx]))
            if INCLUDE_UNEMPLOYMENT:
                unemployment_vals.append(float(row[unemployment_col]))
        for name in dv_accum:
            vals = dv_accum[name]
            record[name] = float(np.mean(vals)) if len(vals) else float('nan')
        for name in skill_accum:
            vals = skill_accum[name]
            record[name] = float(np.mean(vals)) if len(vals) else float('nan')
        if INCLUDE_UNEMPLOYMENT:
            record["unemployment_rate"] = float(np.mean(unemployment_vals)) if len(unemployment_vals) else float('nan')
        records.append(record)
    return pd.DataFrame(records)


def all_dv_names() -> list:
    names  = [name for name, _ in DEP_VAR_SPECS]
    names += [name for name, _ in SKILL_DIST_SPECS]
    if INCLUDE_UNEMPLOYMENT:
        names += ["unemployment_rate"]
    return names


def eta_squared(groups: list) -> float:
    """One-way eta-squared (η²)."""
    all_vals   = np.concatenate(groups)
    grand_mean = all_vals.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total   = sum((v - grand_mean) ** 2 for v in all_vals)
    return ss_between / ss_total if ss_total > 0 else 0.0


def linear_regression(x: np.ndarray, y: np.ndarray) -> dict:
    """
    OLS linear regression of y on x.
    Returns slope, intercept, r_squared, p_value, direction symbol.
    """
    slope, intercept, r, p, se = stats.linregress(x, y)
    direction = "+" if slope > 0 else "-"
    return {
        "slope":     round(slope, 6),
        "intercept": round(intercept, 4),
        "r_squared": round(r ** 2, 4),
        "p_value":   round(p, 4),
        "direction": direction,
    }


def compute_effect_sizes(df: pd.DataFrame) -> pd.DataFrame:
    """Compute η², OLS slope, R², p-value and direction for every DV."""
    group_vals = sorted(df["iv_value"].unique())
    x = df["iv_value"].values.astype(float)
    results = []
    for name in all_dv_names():
        groups = [df.loc[df["iv_value"] == g, name].values for g in group_vals]
        eta2   = eta_squared(groups)
        reg    = linear_regression(x, df[name].values.astype(float))
        results.append({
            "dependent_variable": name,
            "eta_squared":  round(eta2, 4),
            "effect":       "strong" if eta2 > 0.14 else ("moderate" if eta2 > 0.06 else "weak"),
            "slope":        reg["slope"],
            "direction":    reg["direction"],
            "r_squared":    reg["r_squared"],
            "p_value":      reg["p_value"],
        })
    return pd.DataFrame(results).sort_values("eta_squared", ascending=False).reset_index(drop=True)


def plot_effect_sizes(summary: pd.DataFrame, iv_label: str, out_path: str):
    """Bar chart of η² with viridis colors by DV rank, direction arrow + slope annotations."""
    n = len(summary)
    cmap = plt.get_cmap("viridis")
    colors = [cmap(i / max(n - 1, 1)) for i in range(n)]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(summary["dependent_variable"], summary["eta_squared"], color=colors)
    ax.axvline(0.06, color="#cccccc", linestyle="--", linewidth=1.0, label="moderate (0.06)")
    ax.axvline(0.14, color="#888888", linestyle="--", linewidth=1.0, label="strong (0.14)")

    x_max = summary["eta_squared"].max()
    for i, row in summary.iterrows():
        arrow = "▲" if row["direction"] == "+" else "▼"
        ann_color = "#1a9641" if row["direction"] == "+" else "#d7191c"
        label = f"  {arrow} slope={row['slope']:.4g}  R²={row['r_squared']:.3f}"
        ax.text(row["eta_squared"] + x_max * 0.01, i, label,
                va="center", fontsize=7.5, color=ann_color)

    ax.set_xlabel("η² (eta-squared)")
    ax.set_title(f"Effect Size & Direction: '{iv_label}'")
    ax.set_xlim(0, x_max * 1.55)
    ax.legend(fontsize=8)
    ax.set_facecolor("#f8f8f8")
    fig.patch.set_facecolor("white")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_scatter_regression(df: pd.DataFrame, summary: pd.DataFrame,
                            iv_label: str, out_path: str):
    """
    One subplot per DV: scatter of raw runs + OLS regression line.
    IV groups are coloured with viridis. Only DVs with η² > 0.01 are shown.
    """
    dvs = summary[summary["eta_squared"] > 0.01]["dependent_variable"].tolist()
    if not dvs:
        return
    ncols = min(3, len(dvs))
    nrows = (len(dvs) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
    fig.patch.set_facecolor("white")

    x_vals  = df["iv_value"].values.astype(float)
    x_line  = np.linspace(x_vals.min(), x_vals.max(), 100)
    iv_groups = sorted(df["iv_value"].unique())
    cmap    = plt.get_cmap("viridis")
    group_colors = {v: cmap(i / max(len(iv_groups) - 1, 1)) for i, v in enumerate(iv_groups)}

    for ax_idx, dv in enumerate(dvs):
        row_i, col_i = divmod(ax_idx, ncols)
        ax = axes[row_i][col_i]
        ax.set_facecolor("#f8f8f8")
        y_vals = df[dv].values.astype(float)

        for iv_val in iv_groups:
            mask   = df["iv_value"] == iv_val
            jitter = np.random.uniform(-0.02, 0.02, mask.sum()) * (x_vals.max() - x_vals.min())
            ax.scatter(x_vals[mask.values] + jitter, y_vals[mask.values],
                       alpha=0.6, s=20, color=group_colors[iv_val],
                       label=f"IV={iv_val:.4g}")

        slope, intercept, r, p, _ = stats.linregress(x_vals, y_vals)
        y_line    = slope * x_line + intercept
        direction = "▲" if slope > 0 else "▼"
        line_color = "#1a9641" if slope > 0 else "#d7191c"
        ax.plot(x_line, y_line, color=line_color, linewidth=2,
                label=f"{direction} slope={slope:.4g}")

        row_data = summary[summary["dependent_variable"] == dv].iloc[0]
        ax.set_title(f"{dv}\nη²={row_data['eta_squared']:.3f}  R²={r**2:.3f}  p={p:.4f}",
                     fontsize=8)
        ax.set_xlabel(iv_label, fontsize=7)
        ax.set_ylabel(dv, fontsize=7)
        ax.legend(fontsize=6)
        ax.tick_params(labelsize=7)

    for ax_idx in range(len(dvs), nrows * ncols):
        row_i, col_i = divmod(ax_idx, ncols)
        axes[row_i][col_i].set_visible(False)

    fig.suptitle(f"Regression: '{iv_label}'", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


def safe_stem(csv_path: str) -> str:
    return os.path.splitext(os.path.basename(csv_path))[0]


## 4 · Discover CSV files

In [ ]:
# Find all *spreadsheet* CSVs in the input folder
pattern = os.path.join(INPUT_DIR, "*spreadh*eet*.csv")
csv_files = sorted(glob.glob(pattern))

print(f"Found {len(csv_files)} spreadsheet CSV files:\n")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

## 5 · Run batch analysis

In [ ]:
all_summaries = []   # collect η² + regression results from all experiments
failed = []

for csv_path in csv_files:
    stem = safe_stem(csv_path)
    print(f"\n{'─'*60}")
    print(f"Processing: {stem}")

    try:
        raw_param, iv_label = detect_param(csv_path)
        print(f"  IV detected: '{raw_param}' → '{iv_label}'")

        df = load_final_tick_data(csv_path)
        iv_vals = sorted(df["iv_value"].unique())
        print(f"  Runs: {len(df)}  |  IV values: {iv_vals}")

        summary = compute_effect_sizes(df)
        summary.insert(0, "experiment", iv_label)
        all_summaries.append(summary)

        # Effect size bar chart (with direction arrows)
        chart_path = os.path.join(OUTPUT_DIR, f"eta2_{stem}.png")
        plot_effect_sizes(summary.drop(columns="experiment"), iv_label, chart_path)

        # Scatter + regression per DV
        scatter_path = os.path.join(OUTPUT_DIR, f"regression_{stem}.png")
        plot_scatter_regression(df, summary.drop(columns="experiment"), iv_label, scatter_path)

        # Save individual CSV
        csv_out = os.path.join(OUTPUT_DIR, f"eta2_{stem}.csv")
        summary.to_csv(csv_out, index=False)

        top = summary.iloc[0]
        print(f"  Top effect: {top['dependent_variable']} η²={top['eta_squared']} "
              f"({top['effect']})  direction={top['direction']}  slope={top['slope']:.4g}")
        print(f"  Saved → {os.path.basename(chart_path)}")
        print(f"  Saved → {os.path.basename(scatter_path)}")

    except Exception as e:
        import traceback
        print(f"  ⚠ FAILED: {e}")
        traceback.print_exc()
        failed.append((stem, str(e)))

print(f"\n{'='*60}")
print(f"Done. Processed: {len(csv_files) - len(failed)}  |  Failed: {len(failed)}")
if failed:
    for name, err in failed:
        print(f"  ✗ {name}: {err}")

## 6 · Combined summary table

In [ ]:
if all_summaries:
    combined = pd.concat(all_summaries, ignore_index=True)

    # η² pivot
    pivot = combined.pivot_table(
        index="dependent_variable",
        columns="experiment",
        values="eta_squared"
    ).round(4)

    # Slope pivot
    pivot_slope = combined.pivot_table(
        index="dependent_variable",
        columns="experiment",
        values="slope"
    ).round(6)

    # Direction pivot (+ / -)
    pivot_dir = combined.pivot_table(
        index="dependent_variable",
        columns="experiment",
        values="direction",
        aggfunc=lambda x: x.mode()[0],
    )

    print("\n── η² Effect Sizes ──")
    print(pivot.to_string())
    print("\n── OLS Slopes (direction of change per unit IV increase) ──")
    print(pivot_slope.to_string())
    print("\n── Direction summary (+ = DV rises with IV, - = DV falls) ──")
    print(pivot_dir.to_string())
    pivot

## 7 · Combined heatmap

In [ ]:
if all_summaries:
    fig, axes = plt.subplots(
        1, 2,
        figsize=(max(16, len(pivot.columns) * 2.8), max(4, len(pivot.index) * 0.7)),
        gridspec_kw={'width_ratios': [1, 1]}
    )
    fig.patch.set_facecolor("white")

    # ── Left: η² heatmap (viridis) ───────────────────────────────────
    ax = axes[0]
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=0.5)
    plt.colorbar(im, ax=ax, label="η²")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7,
                        color="white" if val < 0.25 else "black")
    ax.set_title("η² Effect Sizes", fontsize=11)
    ax.set_xlabel("Independent Variable")
    ax.set_ylabel("Dependent Variable")

    # ── Right: Direction heatmap (viridis: yellow=+1, purple=-1) ─────
    ax2 = axes[1]
    dir_numeric = pivot_dir.replace({"+": 1, "-": -1}).astype(float)
    mask_weak   = pivot.values < 0.06
    dir_masked  = dir_numeric.values.copy()
    dir_masked[mask_weak] = np.nan

    im2 = ax2.imshow(dir_masked, aspect="auto", cmap="viridis", vmin=-1, vmax=1)
    plt.colorbar(im2, ax=ax2, label="direction (+1 = ▲, −1 = ▼)")
    ax2.set_xticks(range(len(pivot_dir.columns)))
    ax2.set_xticklabels(pivot_dir.columns, rotation=35, ha="right", fontsize=8)
    ax2.set_yticks(range(len(pivot_dir.index)))
    ax2.set_yticklabels(pivot_dir.index, fontsize=9)

    for i in range(len(pivot_dir.index)):
        for j in range(len(pivot_dir.columns)):
            eta2_val = pivot.values[i, j]
            if np.isnan(eta2_val) or eta2_val < 0.06:
                ax2.text(j, i, "~", ha="center", va="center", fontsize=9, color="#aaaaaa")
            else:
                arrow = "▲" if pivot_dir.values[i, j] == "+" else "▼"
                ax2.text(j, i, arrow, ha="center", va="center", fontsize=11, color="white")

    yellow_patch = mpatches.Patch(color=plt.get_cmap('viridis')(1.0), label='DV increases (▲)')
    purple_patch = mpatches.Patch(color=plt.get_cmap('viridis')(0.0), label='DV decreases (▼)')
    grey_patch   = mpatches.Patch(color='#cccccc', label='weak effect (η²<0.06)')
    ax2.legend(handles=[yellow_patch, purple_patch, grey_patch],
               loc='upper right', fontsize=7, framealpha=0.9)
    ax2.set_title("Direction of Change\n[masked where η² < 0.06]", fontsize=11)
    ax2.set_xlabel("Independent Variable")

    plt.suptitle("Sensitivity Analysis — Effect Size & Direction", fontsize=13, y=1.02)
    plt.tight_layout()
    heatmap_path = os.path.join(OUTPUT_DIR, "eta2_heatmap_all_experiments.png")
    plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Heatmap saved → {heatmap_path}")


## 8 · Export combined results

In [ ]:
if all_summaries:
    out_path = os.path.join(OUTPUT_DIR, "eta2_all_experiments.csv")
    combined.to_csv(out_path, index=False)
    print(f"Combined results (η² + slope + direction) → {out_path}")

    pivot_path = os.path.join(OUTPUT_DIR, "eta2_pivot_all_experiments.csv")
    pivot.to_csv(pivot_path)
    print(f"η² pivot table   → {pivot_path}")

    slope_path = os.path.join(OUTPUT_DIR, "slope_pivot_all_experiments.csv")
    pivot_slope.to_csv(slope_path)
    print(f"Slope pivot table → {slope_path}")

    dir_path = os.path.join(OUTPUT_DIR, "direction_pivot_all_experiments.csv")
    pivot_dir.to_csv(dir_path)
    print(f"Direction pivot   → {dir_path}")